# Exploratory Data Analysis: Tech Job Market

**Goal:** Quick overview of roles, salaries, skills, and work types in tech-related postings.

**Data:** Processed job postings (train + test).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12

PROJECT_DIR = Path('..').resolve()
TRAIN_PATH = PROJECT_DIR / 'data' / 'processed' / 'train.csv'
TEST_PATH = PROJECT_DIR / 'data' / 'processed' / 'test.csv'

## 1. Data Loading

In [ ]:
df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

df = pd.concat([df_train, df_test], ignore_index=True)

print("Data loaded:")
print(f"   Train: {len(df_train):,} samples")
print(f"   Test: {len(df_test):,} samples")
print(f"   Total: {len(df):,} samples")
print(f"   Columns: {df.shape[1]}")
print(f"\nAvailable columns: {list(df.columns)}")

In [ ]:
df.head()

## 2. Data Quality

In [ ]:
missing_data = df.isnull().sum()
missing_percentage = (df.isnull().mean() * 100).round(2)

quality_df = pd.DataFrame({
    'Column': df.columns,
    'Missing Values': missing_data.values,
    'Percentage (%)': missing_percentage.values
}).sort_values('Percentage (%)', ascending=False)

print("Data Quality Analysis:\n")
print(quality_df.to_string(index=False))
print(f"\nSamples with salary: {df['max_salary'].notna().sum():,} ({(df['max_salary'].notna().mean()*100):.1f}%)")

In [ ]:
# Check for exact duplicate rows in the dataset
duplicates = df.duplicated().sum()
print(f"Duplicate rows after ETL: {duplicates}")

# Basic statistical summary for max_salary
display(df['max_salary'].describe().round(1))

# 3.  Job Titles Analysis

In [ ]:
# Top 20 most common job titles
title_counts = df['title'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(title_counts)))

bars = ax.barh(title_counts.index[::-1], title_counts.values[::-1], color=colors[::-1])
ax.set_xlabel('Number of Job Postings')
ax.set_title('Top 20 Most Demanded Job Titles in Tech Sector')

# Add value labels to bars
for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.5, bar.get_y() + bar.get_height()/2, 
            f'{int(width)}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"Role diversity: {df['title'].nunique():,} unique titles")

# 4. Job Categories Analysis

In [ ]:
# Target variable distribution (Pre-assigned in ETL)
category_counts = df['job_category'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=category_counts.values, y=category_counts.index, ax=ax, palette='viridis')

ax.set_xlabel('Number of Job Postings')
ax.set_ylabel('')
ax.set_title('Job Categories Distribution')

# Add numbered labels to the bars
for i, v in enumerate(category_counts.values):
    ax.text(v + 10, i, str(v), va='center', fontsize=10)

plt.tight_layout()
plt.show()

print("Percentage of each category (%):")
print((category_counts / len(df) * 100).round(1))

## 5.  Salary analysis

In [ ]:
# Filter postings with valid salary (annual > $20k and < $500k)
df_salary = df[(df['max_salary'].notna()) & (df['max_salary'] > 20000) & (df['max_salary'] < 500000)].copy()

print(f"Postings with valid salary: {len(df_salary):,}")
print("\nMax Salary Statistics:")
print(df_salary['max_salary'].describe().apply(lambda x: f"${x:,.0f}" if x > 100 else f"{x:.2f}"))

In [ ]:
# Aggregate salary statistics by job category
salary_by_category = df_salary.groupby('job_category')['max_salary'].agg(['median', 'mean', 'min', 'max', 'count'])
salary_by_category = salary_by_category.sort_values('median', ascending=False)
salary_by_category.columns = ['Median', 'Mean', 'Minimum', 'Maximum', 'Postings']

# Format as currency
for col in ['Median', 'Mean', 'Minimum', 'Maximum']:
    salary_by_category[col] = salary_by_category[col].apply(lambda x: f"${x:,.0f}")

print("Salary Table by Job Category:")
salary_by_category

## 6.  Work types

In [ ]:
# Work type distribution analysis
print("Distribution by Work Type:\n")

# Calculate counts and percentages
work_type_counts = df['formatted_work_type'].value_counts()
work_type_percentage = df['formatted_work_type'].value_counts(normalize=True) * 100

# Print text summary
for work_type in work_type_counts.index:
    count = work_type_counts[work_type]
    percentage = work_type_percentage[work_type]
    print(f"{work_type:<10} {count:>7,} ({percentage:.1f}%)")

## 7.  Summary

In [ ]:
print("="*60)
print("SUMMARY: Tech Job Market")
print("="*60 + "\n")

print(f"""DATA VOLUME
  • Total postings analyzed: {len(df):,}
  • Unique job titles: {df['title'].nunique():,}
  • Postings with salary: {df['has_salary'].sum():,} ({(df['has_salary'].mean()*100):.1f}%)
""")

print(f"""SALARY INSIGHTS
  • Median salary: ${df_salary['max_salary'].median():,.0f}
  • Average salary: ${df_salary['max_salary'].mean():,.0f}
  • Salary range: ${df_salary['max_salary'].min():,.0f} - ${df_salary['max_salary'].max():,.0f}
""")

print("TOP 5 MOST DEMANDED ROLES")
for i, (role, count) in enumerate(df['job_category'].value_counts().head(5).items(), 1):
    print(f"  {i}. {role}: {count:,} postings")